In [304]:
import pandas as pd
import numpy as np

customers = pd.DataFrame({
    'customer_id': [101, 102, 103, 104],
    'tier': ['Gold', 'Silver', 'Gold', 'Bronze']
})

orders = pd.DataFrame({
    'order_id':   [1, 2, 3, 4, 5, 6, 7],
    'customer_id':[101, 102, 101, 103, 101, 102, 105],  # note: 105 isn't in customers!
    'order_date': pd.to_datetime([
        '2026-01-05', '2026-01-10', '2026-02-03',
        '2026-02-14', '2026-03-01', '2026-03-10', '2026-03-15'
    ]),
    'amount': [200, 120, 350, 500, 150, 180, 90]
})

targets = pd.DataFrame({
    'tier': ['Gold', 'Silver', 'Bronze'],
    'Jan':  [300, 150, 100],
    'Feb':  [700, 200, 100],
    'Mar':  [400, 200, 100]
})

##### 1. Tier Spend Report: Using customers and orders, find the total spend and order count for every customer tier (including tiers with 0 orders). Also identify any order whose customer_id doesn't exist in customers.
##### 2.Actual vs. Target by Month: Reshape targets from monthly columns into rows, then compare each tier's monthly actual spend against its monthly target (pct_achieved).
##### 3.Customer Order History: Sort orders by customer_id and order_date, then add 3 columns per customer:
    - 1. order_num (1st, 2nd, 3rd order for that customer)
    - 2. prev_amount (their previous order's amount)
    - 3. rolling_2_avg (average of their current and previous order)

In [2]:
print(customers)
print(orders)
print(targets)

   customer_id    tier
0          101    Gold
1          102  Silver
2          103    Gold
3          104  Bronze
   order_id  customer_id order_date  amount
0         1          101 2026-01-05     200
1         2          102 2026-01-10     120
2         3          101 2026-02-03     350
3         4          103 2026-02-14     500
4         5          101 2026-03-01     150
5         6          102 2026-03-10     180
6         7          105 2026-03-15      90
     tier  Jan  Feb  Mar
0    Gold  300  700  400
1  Silver  150  200  200
2  Bronze  100  100  100


#### Tier Spend Report

In [138]:
merged_df = pd.merge(customers, orders, how = 'outer', on='customer_id')
merged_df

,customer_id,tier,order_id,order_date,amount
0,101,Gold,1.0,2026-01-05,200.0
1,101,Gold,3.0,2026-02-03,350.0
2,101,Gold,5.0,2026-03-01,150.0
3,102,Silver,2.0,2026-01-10,120.0
4,102,Silver,6.0,2026-03-10,180.0
5,103,Gold,4.0,2026-02-14,500.0
6,104,Bronze,NaN,NaT,NaN
7,105,NaN,7.0,2026-03-15,90.0


In [139]:
tier_group = merged_df.groupby('tier')

tier_group['amount'].agg(['sum','count'])

,sum,count
tier,,
Bronze,0.0,0
Gold,1200.0,4
Silver,300.0,2


In [140]:
merged_df[merged_df['tier'].isna()]

,customer_id,tier,order_id,order_date,amount
7,105,NaN,7.0,2026-03-15,90.0


##### Actual vs Target by Month

In [141]:
#monthly target by tier
target_series = targets.set_index('tier').unstack()
target_series

     tier  
Jan  Gold      300
     Silver    150
     Bronze    100
Feb  Gold      700
     Silver    200
     Bronze    100
Mar  Gold      400
     Silver    200
     Bronze    100
dtype: int64

In [142]:
target_series = target_series.swaplevel().sort_index()
target_series

tier       
Bronze  Feb    100
        Jan    100
        Mar    100
Gold    Feb    700
        Jan    300
        Mar    400
Silver  Feb    200
        Jan    150
        Mar    200
dtype: int64

In [143]:
#monthly spend by tier
merged_df

,customer_id,tier,order_id,order_date,amount
0,101,Gold,1.0,2026-01-05,200.0
1,101,Gold,3.0,2026-02-03,350.0
2,101,Gold,5.0,2026-03-01,150.0
3,102,Silver,2.0,2026-01-10,120.0
4,102,Silver,6.0,2026-03-10,180.0
5,103,Gold,4.0,2026-02-14,500.0
6,104,Bronze,NaN,NaT,NaN
7,105,NaN,7.0,2026-03-15,90.0


In [184]:
# merged_df_new = merged_df[~merged_df.order_date.isna()]
merged_df_new = merged_df.copy()
merged_df_new = merged_df_new[~merged_df_new.order_date.isna()]

In [185]:
merged_df_new.sort_values('order_date', inplace=True)

In [186]:
merged_df_new['month_name'] = merged_df_new['order_date'].dt.strftime('%b')
merged_df_new['month_name'] = merged_df_new['month_name'].str.strip()
merged_df_new

,customer_id,tier,order_id,order_date,amount,month_name
0,101,Gold,1.0,2026-01-05,200.0,Jan
3,102,Silver,2.0,2026-01-10,120.0,Jan
1,101,Gold,3.0,2026-02-03,350.0,Feb
5,103,Gold,4.0,2026-02-14,500.0,Feb
2,101,Gold,5.0,2026-03-01,150.0,Mar
4,102,Silver,6.0,2026-03-10,180.0,Mar
7,105,NaN,7.0,2026-03-15,90.0,Mar


In [162]:
# merged_df['order_date'] = pd.to_datetime(merged_df[~merged_df.order_date.isna()].order_date, format ='%Y-%m-%d %I-%p')
# merged_df.set_index('order_date')
merged_df_new

,customer_id,tier,order_id,order_date,amount,month_name
0,101,Gold,1.0,2026-01-05,200.0,Jan
3,102,Silver,2.0,2026-01-10,120.0,Jan
1,101,Gold,3.0,2026-02-03,350.0,Feb
5,103,Gold,4.0,2026-02-14,500.0,Feb
2,101,Gold,5.0,2026-03-01,150.0,Mar
4,102,Silver,6.0,2026-03-10,180.0,Mar
7,105,NaN,7.0,2026-03-15,90.0,Mar


In [187]:
merged_df_new.set_index('order_date', inplace=True)
# merged_df.sort_index()

In [190]:
tier_group = merged_df_new.groupby(['tier', 'month_name'])

In [194]:
spend_series = tier_group['amount'].sum()
spend_series

tier    month_name
Gold    Feb           850.0
        Jan           200.0
        Mar           150.0
Silver  Jan           120.0
        Mar           180.0
Name: amount, dtype: float64

In [211]:
target_series_df = pd.DataFrame(target_series)
spend_series_df = pd.DataFrame(spend_series)

In [216]:
target_series_df.index.names = ['tier', 'month_name']
target_series_df.rename(columns = {0:'target_amount'}, inplace=True)

In [217]:
target_series_df

target_amount
tier   month_name               
Bronze Feb                   100
       Jan                   100
       Mar                   100
Gold   Feb                   700
       Jan                   300
       Mar                   400
Silver Feb                   200
       Jan                   150
       Mar                   200

In [218]:
spend_series_df

amount
tier   month_name        
Gold   Feb          850.0
       Jan          200.0
       Mar          150.0
Silver Jan          120.0
       Mar          180.0

In [226]:
target_spent_df = pd.merge(target_series_df, spend_series_df, left_index = True, right_index=True, how = 'left')
target_spent_df['amount'] = target_spent_df['amount'].fillna(0)
target_spent_df

target_amount  amount
tier   month_name                       
Bronze Feb                   100     0.0
       Jan                   100     0.0
       Mar                   100     0.0
Gold   Feb                   700   850.0
       Jan                   300   200.0
       Mar                   400   150.0
Silver Feb                   200     0.0
       Jan                   150   120.0
       Mar                   200   180.0

In [227]:
target_spent_df['pct_achieved'] = target_spent_df['amount']/target_spent_df['target_amount']*100
target_spent_df

target_amount  amount  pct_achieved
tier   month_name                                     
Bronze Feb                   100     0.0      0.000000
       Jan                   100     0.0      0.000000
       Mar                   100     0.0      0.000000
Gold   Feb                   700   850.0    121.428571
       Jan                   300   200.0     66.666667
       Mar                   400   150.0     37.500000
Silver Feb                   200     0.0      0.000000
       Jan                   150   120.0     80.000000
       Mar                   200   180.0     90.000000

##### Customer Order History

##### 3.Customer Order History: Sort orders by customer_id and order_date, then add 3 columns per customer:
    - 1. order_num (1st, 2nd, 3rd order for that customer)
    - 2. prev_amount (their previous order's amount)
    - 3. rolling_2_avg (average of their current and previous order)

In [305]:
orders.sort_values(by=['customer_id', 'order_date'], ascending = [True, True], inplace=True)
orders

,order_id,customer_id,order_date,amount
0,1,101,2026-01-05,200
2,3,101,2026-02-03,350
4,5,101,2026-03-01,150
1,2,102,2026-01-10,120
5,6,102,2026-03-10,180
3,4,103,2026-02-14,500
6,7,105,2026-03-15,90


In [306]:
orders['dummy_col'] = 1
orders_gr = orders.groupby('customer_id')

order_num = orders_gr['dummy_col'].cumsum()


In [307]:
order_num = pd.DataFrame(order_num)
order_num

,dummy_col
0,1
2,2
4,3
1,1
5,2
3,1
6,1


In [308]:
orders = pd.merge(orders, order_num, left_index=True, right_index = True)
orders.drop(columns = 'dummy_col_x', inplace = True)
orders.rename(columns = {'dummy_col_y':'order_num'}, inplace=True)
orders

,order_id,customer_id,order_date,amount,order_num
0,1,101,2026-01-05,200,1
2,3,101,2026-02-03,350,2
4,5,101,2026-03-01,150,3
1,2,102,2026-01-10,120,1
5,6,102,2026-03-10,180,2
3,4,103,2026-02-14,500,1
6,7,105,2026-03-15,90,1


In [309]:
rolling_avg = orders_gr['amount'].rolling(2, min_periods=1).mean()
rolling_avg = pd.DataFrame(rolling_avg)
rolling_avg

amount
customer_id          
101         0   200.0
            2   275.0
            4   250.0
102         1   120.0
            5   150.0
103         3   500.0
105         6    90.0

In [310]:
rolling_avg.index.names = ['customer_id', 'index_col']

In [291]:
rolling_avg

amount
customer_id index_col        
101         0           200.0
            2           275.0
            4           250.0
102         1           120.0
            5           150.0
103         3           500.0
105         6            90.0

In [311]:
orders.index.name = 'index_col'

In [293]:
orders


,order_id,customer_id,order_date,amount,order_num
index_col,,,,,
0,1,101,2026-01-05,200,1
2,3,101,2026-02-03,350,2
4,5,101,2026-03-01,150,3
1,2,102,2026-01-10,120,1
5,6,102,2026-03-10,180,2
3,4,103,2026-02-14,500,1
6,7,105,2026-03-15,90,1


In [312]:
orders = pd.merge(orders, rolling_avg, left_index = True, right_index = True)

In [313]:
orders =  orders.rename(columns = {'amount_x':'amount', 'amount_y':'rolling_2_avg'})
orders

order_id  customer_id order_date  amount  order_num  \
customer_id index_col                                                        
101         0                 1          101 2026-01-05     200          1   
            2                 3          101 2026-02-03     350          2   
            4                 5          101 2026-03-01     150          3   
102         1                 2          102 2026-01-10     120          1   
            5                 6          102 2026-03-10     180          2   
103         3                 4          103 2026-02-14     500          1   
105         6                 7          105 2026-03-15      90          1   

                       rolling_2_avg  
customer_id index_col                 
101         0                  200.0  
            2                  275.0  
            4                  250.0  
102         1                  120.0  
            5                  150.0  
103         3                  500.0  
105         6                   90.0

In [314]:
orders = orders.droplevel('customer_id')
orders

,order_id,customer_id,order_date,amount,order_num,rolling_2_avg
index_col,,,,,,
0,1,101,2026-01-05,200,1,200.0
2,3,101,2026-02-03,350,2,275.0
4,5,101,2026-03-01,150,3,250.0
1,2,102,2026-01-10,120,1,120.0
5,6,102,2026-03-10,180,2,150.0
3,4,103,2026-02-14,500,1,500.0
6,7,105,2026-03-15,90,1,90.0


In [323]:
orders_set = orders[['order_num', 'amount']]
orders_set

,order_num,amount
index_col,,
0,1,200
2,2,350
4,3,150
1,1,120
5,2,180
3,1,500
6,1,90


In [330]:
pd.merge(orders, orders[['customer_id','order_num', 'amount']], how = 'left', left_on=['customer_id','order_num'], right_on=['customer_id',orders['order_num']+1])

,order_id,order_num,customer_id,order_date,amount_x,order_num_x,rolling_2_avg,order_num_y,amount_y
0,1,1,101,2026-01-05,200,1,200.0,NaN,NaN
1,3,2,101,2026-02-03,350,2,275.0,1.0,200.0
2,5,3,101,2026-03-01,150,3,250.0,2.0,350.0
3,2,1,102,2026-01-10,120,1,120.0,NaN,NaN
4,6,2,102,2026-03-10,180,2,150.0,1.0,120.0
5,4,1,103,2026-02-14,500,1,500.0,NaN,NaN
6,7,1,105,2026-03-15,90,1,90.0,NaN,NaN


In [316]:
help(pd.merge)

Help on function merge in module pandas:

merge(left: 'DataFrame | Series', right: 'DataFrame | Series', how: 'MergeHow' = 'inner', on: 'IndexLabel | AnyArrayLike | None' = None, left_on: 'IndexLabel | AnyArrayLike | None' = None, right_on: 'IndexLabel | AnyArrayLike | None' = None, left_index: 'bool' = False, right_index: 'bool' = False, sort: 'bool' = False, suffixes: 'Suffixes' = ('_x', '_y'), copy: 'bool | lib.NoDefault' = <no_default>, indicator: 'str | bool' = False, validate: 'str | None' = None) -> 'DataFrame'
    Merge DataFrame or named Series objects with a database-style join.
    
    A named Series object is treated as a DataFrame with a single named column.
    
    The join is done on columns or indexes. If joining columns on
    columns, the DataFrame indexes *will be ignored*. Otherwise if joining indexes
    on indexes or indexes on a column or columns, the index will be passed on.
    When performing a cross merge, no column specifications to merge on are
    allowe

In [331]:
orders_gr.cumcount() + 1

0    1
2    2
4    3
1    1
5    2
3    1
6    1
dtype: int64

In [332]:
orders_gr.cumcount() 

0    0
2    1
4    2
1    0
5    1
3    0
6    0
dtype: int64

In [333]:
orders_gr['amount'].shift(1)

0      NaN
2    200.0
4    350.0
1      NaN
5    120.0
3      NaN
6      NaN
Name: amount, dtype: float64

In [334]:
orders_gr['amount'].rolling(2, min_periods=1).mean().reset_index(level=0, drop=True)

0    200.0
2    275.0
4    250.0
1    120.0
5    150.0
3    500.0
6     90.0
Name: amount, dtype: float64